In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import optuna

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool

import warnings
warnings.filterwarnings('ignore')

SEED = 42

In [ ]:
ROOT = Path.cwd().parent.resolve()

TRAIN = ROOT / "data/train.csv"
TEST = ROOT / "data/test.csv"
OUT = ROOT / "outputs"
Path.mkdir(OUT , exist_ok=True)
OUT = OUT / "catboost"
Path.mkdir(OUT , exist_ok=True)

train_df = pd.read_csv(str(TRAIN), sep=',')
test_df = pd.read_csv(str(TEST), sep=',')

X = train_df.drop(columns=["id", "Heart Disease"])
le = LabelEncoder()
y = le.fit_transform(train_df["Heart Disease"])
X_test = test_df.drop(columns="id")

cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

In [ ]:
def objective(trial):
    params = {
        "depth": trial.suggest_int("depth", 3, 8),
        "iterations": trial.suggest_int("iterations", 500, 1500),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.5, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10)
    }

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    oof = np.zeros(len(X))

    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
    
        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        val_pool   = Pool(X_val, y_val, cat_features=cat_features)

        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            task_type="GPU",
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            **params
        )

        model.fit(train_pool, eval_set=val_pool, use_best_model=True)

        oof[val_idx] = model.predict_proba(val_pool)[:, 1]
    
    return roc_auc_score(y, oof)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED) 
)

study.optimize(objective, n_trials=25)

In [ ]:
best_params = study.best_params
seeds = [1, 42, 157, 2531]
tests = []
oofs = []
for seed in seeds:
    params = {**best_params, "random_seed": seed}

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

    oof_seed = np.zeros(len(X))
    test_seed = np.zeros(len(X_test))

    test_pool = Pool(X_test, cat_features=cat_features)
    
    for train_idx, val_idx in skf.split(X, y):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y[train_idx], y[val_idx]
    
        train_pool = Pool(X_train, y_train, cat_features=cat_features)
        val_pool   = Pool(X_val, y_val, cat_features=cat_features)

        model = CatBoostClassifier(
            loss_function="Logloss",
            eval_metric="AUC",
            task_type="GPU",
            verbose=False,
            allow_writing_files=False,
            **params
        )

        model.fit(train_pool, eval_set=val_pool, use_best_model=True)

        oof_seed[val_idx] = model.predict_proba(val_pool)[:, 1]
        test_seed += model.predict_proba(test_pool)[:, 1] / skf.n_splits

    oofs.append(oof_seed)
    tests.append(test_seed)

In [ ]:
final_oofs = np.vstack(oofs).mean(axis=0)
final_preds = np.vstack(tests).mean(axis=0)

np.save(OUT / "cat_oof.npy", final_oofs)
np.save(OUT / "cat_test.npy", final_preds)

submission = pd.DataFrame({
    "id": test_df["id"],
    "Heart Disease": final_preds
})
submission.to_csv(OUT / "cat_submission.csv", index=False)
submission